In [14]:
pip install tqdm

Note: you may need to restart the kernel to use updated packages.


In [5]:
import json
import hashlib
from pathlib import Path
from tqdm import tqdm

source = Path("data/All_Beauty.jsonl")
output = Path("data/raw/reviews.jsonl")
checkpoint = Path("checkpoint.json")

output.parent.mkdir(parents=True, exist_ok=True)

# 마지막 처리 위치 읽기
if checkpoint.exists():
    start = json.loads(
        checkpoint.read_text()
    )["last_line"]
else:
    start = 0

# 기존 저장 데이터의 중복 키 읽기
seen = set()
if output.exists():
    with output.open("r", encoding="utf-8") as f:
        for line in f:
            seen.add(hashlib.sha256(
                line.rstrip("\r\n").encode("utf-8")
            ).hexdigest())


# 새로운 데이터 수집

from tqdm import tqdm

# 전체 데이터 개수 확인
with source.open("r", encoding="utf-8") as f:
    total_lines = sum(1 for line in f)

# 진행률 표시하면서 데이터 수집
with source.open("r", encoding="utf-8") as src:
    for number, line in tqdm(
        enumerate(src, start=1),
        total=total_lines,
        desc="아마존 리뷰 수집",
        unit="건"
    ):
        if number <= start:
            continue

        if line.strip():
            key = hashlib.sha256(
                line.rstrip("\r\n").encode("utf-8")
            ).hexdigest()

            if key not in seen:
                with output.open("a", encoding="utf-8") as dst:
                    dst.write(line.rstrip("\r\n") + "\n")
                seen.add(key)

        checkpoint.write_text(
            json.dumps({"last_line": number})
        )

아마존 리뷰 수집: 100%|█████████████████████████████████████████████████████████████████████████████| 701528/701528 [01:31<00:00, 7674.61건/s]


In [13]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/Users/steven/Documents/메타코드_Local/Live 프로젝트"
)

module_file = PROJECT_ROOT / "jobs" / "amazon_reviews_etl.py"

print("노트북 작업 위치:", Path.cwd())
print("모듈 파일 존재:", module_file.is_file())

assert module_file.is_file(), (
    "노트북 환경에서 이 경로에 접근할 수 없습니다. "
    "Docker나 원격 Jupyter라면 해당 환경 내부 경로를 사용해야 합니다."
)

sys.path.insert(0, str(module_file.parent))

from amazon_reviews_etl import INPUT_SCHEMA, parse_hash_columns, transform
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("amazon-reviews-notebook")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

raw = (
    spark.read
    .schema(INPUT_SCHEMA)
    .option("mode", "FAILFAST")
    .json(str(PROJECT_ROOT / "data" / "All_Beauty.jsonl"))
)

cleaned = transform(
    raw,
    parse_hash_columns("user_id,product_id,review_text"),
)

cleaned.show(truncate=False)
print("정제 후 행 수:", cleaned.count())  # 기대 결과: 3

노트북 작업 위치: /Users/steven/Documents/메타코드_Local/Live 프로젝트
모듈 파일 존재: True


+----------------------------------------------------------------+----------------------------+----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+-------------------------------------------------------+-----------------------+-----------------+
|review_id                                                       |user_id                     |product_id|review_text                                                                      

[Stage 3:=======================================>                   (2 + 1) / 3]

정제 후 행 수: 693547


In [11]:
!pip install pyspark

  Using cached pyspark-4.2.0.tar.gz (450.1 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached py4j-0.10.9.9-py2.py3-none-any.whl.metadata (1.3 kB)
Using cached py4j-0.10.9.9-py2.py3-none-any.whl (203 kB)
  Created wheel for pyspark: filename=pyspark-4.2.0-py2.py3-none-any.whl size=450798674 sha256=49efadb92993f8de0975f0246d4c64ada9dc97de10c867e3d928c0cef7266f9d
  Stored in directory: /Users/steven/Library/Caches/pip/wheels/61/69/04/e7d825c33d54d9b6e7a1b94c56998f54832c38b64553603219
Successfully built pyspark
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pyspark]━━━ 1/2 [pyspark]
